# Estilos de juego: clustering no supervisado — Top 5 ligas 2025-26

Primera experimentación de ML no supervisado sobre los datos de equipo: en vez de
comparar equipos eje por eje (como en `eda_teams.ipynb`), acá buscamos **arquetipos
de estilo de juego** que emergen de las 19 métricas de estilo combinadas, sin
mirar la liga a la que pertenece cada equipo.

Tres miradas independientes sobre la misma pregunta ("¿qué equipos juegan parecido?"):

1. **PCA** — reduce las métricas a un puñado de componentes interpretables (el
   "vocabulario de estilo").
2. **K-means** — clusters duros sobre esos componentes.
3. **GMM** — mismos clusters pero con membership probabilístico (quién es híbrido).
4. **Red de similitud + comunidades** — sin asumir forma esférica de cluster,
   una tercera validación independiente.

La tabla comparativa cruza los tres métodos: cuando coinciden, el grupo es
sólido; cuando no, el equipo vive en la frontera entre dos.

> **Cómo terminó esto** (conviene saberlo antes de empezar, porque cambia cómo se
> lee todo lo demás): el ejercicio funcionó, pero descubrió que **estaba midiendo
> otra cosa**. El eje que domina el agrupamiento correlaciona `+0.93` con los
> puntos por partido — una variable que se excluyó a propósito del modelo. Es
> decir: con datos públicos de fbref, lo que más separa a los equipos europeos no
> es *cómo* juegan sino *qué tan buenos son*. La **sección 5.1** lo demuestra, y
> la **sección 9** rehace el ejercicio quitando el nivel para ver qué queda.
>
> El recorrido con el error se deja completo a propósito: el diagnóstico es más
> útil que un resultado limpio del que no se sabe cómo se llegó.

> **Sobre el color en este notebook**: el color significa siempre **grupo de
> estilo**, nunca liga — por eso no se usa la paleta de ligas del resto del
> proyecto (mezclar los dos significados haría que un punto rojo pareciera "de
> la Bundesliga"). La liga de cada equipo está en el *hover* y en las tablas.
> Todos los gráficos son interactivos: pasa el mouse por encima para ver de qué
> equipo se trata.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

from viz_theme import (LEAGUE_ORDER, CLUSTER_COLORS, SEQUENTIAL_BLUE, DIVERGING, INK,
                        apply_plotly_theme, render_plot)

# opaque_surface=True: en el notebook el gráfico se dibuja sobre su propia
# superficie clara en vez de heredar el fondo del visor (VS Code oscuro,
# Jupyter claro). El sitio sigue usando el default transparente.
apply_plotly_theme(opaque_surface=True)
pd.set_option('display.max_columns', 60)
RNG = 42

# escala divergente (z-score de perfil de cluster): dos polos + gris en el cero
DIVERGING_SCALE = [[0.0, DIVERGING['neg']], [0.5, DIVERGING['mid']], [1.0, DIVERGING['pos']]]

## 1. Preparación

### Carga y merge (mismo criterio que `eda_teams.ipynb`)

Los 5 CSV vienen con header de dos filas (grupo + métrica) y sin columna de liga
explícita; se aplana el header y se asigna `liga` por bloques de filas (orden
verificado: Bundesliga, Serie A, Ligue 1, La Liga, Premier League).

In [2]:
SEASON = '2025-26'
DATA_DIR = f'../data/{SEASON}'

RAW_FILES = {
    'ov': 'leagues_overall.csv',
    'sh': 'leagues_shoot.csv',
    'pt': 'leagues_playtime.csv',
    'ms': 'leagues_misc.csv',
    'gk': 'leagues_gk.csv',
}

LEAGUE_BLOCKS = [
    ('Bundesliga', 18),
    ('Serie A', 20),
    ('Ligue 1', 18),
    ('La Liga', 20),
    ('Premier League', 20),
]
liga_col = [liga for liga, n in LEAGUE_BLOCKS for _ in range(n)]


def flatten_columns(raw):
    cols = []
    for top, bot in raw.columns:
        top = '' if str(top).startswith('Unnamed') else str(top).strip()
        bot = str(bot).strip()
        cols.append(f'{top}_{bot}' if top else bot)
    raw.columns = cols
    return raw


frames = {}
for tag, fname in RAW_FILES.items():
    raw = pd.read_csv(f'{DATA_DIR}/{fname}', header=[0, 1])
    raw = flatten_columns(raw)
    assert len(raw) == len(liga_col), f'{fname}: {len(raw)} filas, se esperaban {len(liga_col)}'
    raw['liga'] = liga_col
    stat_cols = [c for c in raw.columns if c not in ('Squad', 'liga')]
    raw = raw.rename(columns={c: f'{tag}_{c}' for c in stat_cols})
    frames[tag] = raw

df = frames['ov']
for tag in ['sh', 'pt', 'ms', 'gk']:
    df = df.merge(frames[tag].drop(columns=['liga']), on='Squad', how='left')

df['liga'] = pd.Categorical(df['liga'], categories=LEAGUE_ORDER, ordered=True)

# Las columnas `90s` de estos archivos son partidos jugados del EQUIPO (se
# verificó: coinciden exacto con MP en los 96 equipos, diferencia máxima 0.0),
# así que dividir por ellas da tasas por partido — comparables entre equipos
# aunque lleven distinto número de jornadas disputadas (van de 33 a 38).
for col in ['ms_Performance_Fls', 'ms_Performance_Off', 'ms_Performance_Int',
            'ms_Performance_TklW', 'ms_Performance_CrdY', 'ms_Performance_CrdR',
            'ms_Performance_Crs', 'ms_Performance_Fld']:
    df[col.replace('ms_Performance_', 'p90_')] = df[col] / df['ms_90s']
df['p90_SoTA'] = df['gk_Performance_SoTA'] / df['gk_Playing Time_90s']
df = df.copy()  # desfragmenta tras insertar columnas de a una

assert (df['ms_90s'] - df['ov_Playing Time_MP']).abs().max() == 0, '90s no es partidos jugados'
print(df.shape, '| partidos jugados:', df['ov_Playing Time_MP'].min(), 'a', df['ov_Playing Time_MP'].max())
df[['Squad', 'liga']].groupby('liga', observed=True).size()

(96, 95) | partidos jugados: 33 a 38


liga
Bundesliga        18
Serie A           20
Ligue 1           18
La Liga           20
Premier League    20
dtype: int64

### Selección de features de estilo

Criterio: **estilo, no contexto ni resultado.**

- **Fuera — contexto**: edad de plantilla, `# Pl`, minutos/titularidades/rotación
  (`Min`, `MP`, `Starts`, `Mn/MP`, `Min%`, `Compl`, `Subs`, `Mn/Sub`, `unSub`) —
  describen la gestión de plantilla, no cómo juega el equipo.
- **Fuera — resultado, no estilo**: `PPM`, `+/-`, `+/-90`, `onG`, `onGA`, W/D/L,
  `CS`/`CS%` — miden qué tan bueno es el equipo (posición en la tabla), y eso
  contaminaría el clustering: dos equipos pueden jugar exactamente igual y tener
  puntos muy distintos porque uno ejecuta mejor. Igual con penales (`PK*`,
  `Penalty Kicks_*`): muestra chica y ruidosa, no es un rasgo de estilo.
- **Fuera — conteos crudos redundantes**: si ya existe la versión por 90' o en
  porcentaje del mismo evento (`Sh` vs. `Sh/90`, `Gls` vs. `Gls/90`), se
  mantiene solo la tasa — el conteo crudo agrega la misma señal pegada a los
  minutos jugados, doble-contando esa dimensión en el PCA.
- **Dentro**: todo lo demás por 90' o en %, que sí describe *cómo* juega el
  equipo — posesión, generación y definición de tiro, faltas/fuera de juego/
  centros como proxy de intensidad y ataque por banda, tackles/intercepciones
  como proxy de presión, y del lado de portería GA90/Save%/SoTA-90 como qué
  tanto expone y resuelve la defensa (no CS%, que es resultado).

> ⚠️ **Spoiler: este criterio tiene una grieta, y es el hallazgo principal del
> notebook.** Dejamos fuera `+/-` por ser resultado… pero dejamos dentro
> `Goles/90` y `Goles recibidos/90`, que son *exactamente sus dos componentes*.
> Lo que echamos por la puerta vuelve por la ventana. En la **sección 5.1** se
> mide el daño y en la **sección 9** se corrige. Se deja el recorrido completo a
> propósito, porque el error es más instructivo que el resultado limpio.

**Dos límites de los datos que conviene tener presentes**: la temporada está en
curso y los equipos no llevan las mismas jornadas (de 33 a 38) — por eso todo va
en tasas por partido y no en totales. Y hay features con muy poca señal: en
`Rojas/90` la variación entre equipos apenas supera el ruido estadístico
esperable (razón señal/ruido ≈ 1.2 contra el ruido de Poisson de un evento que
ocurre ~3.6 veces por temporada), así que tras estandarizar aporta casi solo
ruido, con el mismo peso que la posesión. Se mantiene en esta primera pasada
para no cambiar el criterio a mitad de camino, y se elimina en la sección 9.

In [3]:
FEATURES = {
    'ov_Poss': 'Posesión (%)',
    'ov_Per 90 Minutes_Gls': 'Goles/90',
    'ov_Per 90 Minutes_Ast': 'Asistencias/90',
    'sh_Standard_Sh/90': 'Tiros/90',
    'sh_Standard_SoT/90': 'Tiros a puerta/90',
    'sh_Standard_SoT%': 'Precisión de tiro (%)',
    'sh_Standard_G/Sh': 'Goles por tiro',
    'sh_Standard_G/SoT': 'Goles por tiro a puerta',
    'p90_CrdY': 'Amarillas/90',
    'p90_CrdR': 'Rojas/90',
    'p90_Fls': 'Faltas cometidas/90',
    'p90_Fld': 'Faltas recibidas/90',
    'p90_Off': 'Fuera de juego/90',
    'p90_Crs': 'Centros/90',
    'p90_Int': 'Intercepciones/90',
    'p90_TklW': 'Tackles ganados/90',
    'gk_Performance_GA90': 'Goles recibidos/90',
    'gk_Performance_Save%': 'Atajadas (%)',
    'p90_SoTA': 'Tiros a puerta recibidos/90',
}
feature_cols = list(FEATURES.keys())

assert df[feature_cols].isna().sum().sum() == 0, 'hay NaN en las features de estilo'
print(len(feature_cols), 'features de estilo sobre', df.shape[1], 'columnas totales')
df[['Squad', 'liga'] + feature_cols].head(3)

19 features de estilo sobre 95 columnas totales


,Squad,liga,ov_Poss,ov_Per 90 Minutes_Gls,ov_Per 90 Minutes_Ast,sh_Standard_Sh/90,sh_Standard_SoT/90,sh_Standard_SoT%,sh_Standard_G/Sh,sh_Standard_G/SoT,p90_CrdY,p90_CrdR,p90_Fls,p90_Fld,p90_Off,p90_Crs,p90_Int,p90_TklW,gk_Performance_GA90,gk_Performance_Save%,p90_SoTA
0,Augsburg,Bundesliga,44.0,1.32,0.79,13.09,4.62,35.3,0.09,0.26,2.411765,0.029412,11.029412,10.911765,2.117647,18.411765,8.500000,9.852941,1.79,63.3,4.882353
1,Bayern Munich,Bundesliga,67.5,3.50,2.68,19.21,8.50,44.3,0.17,0.37,1.705882,0.058824,8.794118,8.558824,2.441176,16.147059,8.411765,8.323529,1.06,65.7,3.088235
2,Dortmund,Bundesliga,53.3,2.03,1.50,13.09,4.79,36.6,0.14,0.39,1.852941,0.058824,11.647059,9.617647,1.176471,19.676471,7.970588,9.117647,1.00,72.1,3.588235


### Escalado

`StandardScaler` es imprescindible acá: `Poss` vive en 0-100, `G/Sh` en 0-0.3,
`Goles recibidos/90` en 0-3 — sin escalar, el PCA quedaría dominado por las
columnas de mayor magnitud (posesión) sin importar cuánta varianza *real*
explican. Con z-score, cada feature entra al PCA con la misma escala.

In [4]:
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df[feature_cols]),
                  columns=feature_cols, index=df['Squad'])
X.describe().loc[['mean', 'std']].round(2)

,ov_Poss,ov_Per 90 Minutes_Gls,ov_Per 90 Minutes_Ast,sh_Standard_Sh/90,sh_Standard_SoT/90,sh_Standard_SoT%,sh_Standard_G/Sh,sh_Standard_G/SoT,p90_CrdY,p90_CrdR,p90_Fls,p90_Fld,p90_Off,p90_Crs,p90_Int,p90_TklW,gk_Performance_GA90,gk_Performance_Save%,p90_SoTA
mean,0.00,0.00,-0.00,0.00,-0.00,-0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00
std,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01,1.01


## 2. PCA

Reducimos las 19 features escaladas a un puñado de componentes principales que
capturen la mayor parte de la varianza — esos componentes son los que se usan
después para K-means, GMM y la red de similitud (en vez de las 19 features
originales, correlacionadas entre sí).

In [5]:
pca_full = PCA(random_state=RNG).fit(X)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

n_components = int(np.searchsorted(cumulative, 0.80) + 1)
print(f'Componentes para llegar a 80% de varianza explicada: {n_components}')

# Un solo PCA para todo lo que sigue: los `n_components` que explican el 80%.
# El biplot usa las 2 primeras columnas de este mismo ajuste (se verificó que
# son idénticas a las de un PCA de 2 componentes, diferencia máxima 0.0), así
# que biplot y clustering viven en exactamente el mismo espacio.
pca = PCA(n_components=n_components, random_state=RNG)
scores = pca.fit_transform(X)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
pc1_var, pc2_var = pca.explained_variance_ratio_[:2]

pd.DataFrame({
    'Componente': [f'PC{i}' for i in range(1, len(explained) + 1)],
    'Varianza individual': (explained * 100).round(1),
    'Varianza acumulada': (cumulative * 100).round(1),
}).head(10)

Componentes para llegar a 80% de varianza explicada: 7


,Componente,Varianza individual,Varianza acumulada
0,PC1,38.2,38.2
1,PC2,12.7,50.9
2,PC3,10.7,61.7
3,PC4,7.3,68.9
4,PC5,5.4,74.3
5,PC6,4.9,79.2
6,PC7,4.6,83.8
7,PC8,3.4,87.2
8,PC9,3.0,90.2
9,PC10,2.5,92.8


In [6]:
x_pc = np.arange(1, len(explained) + 1)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=x_pc, y=explained * 100, name='Varianza individual',
    marker=dict(color=SEQUENTIAL_BLUE[2], line=dict(width=0)),
    hovertemplate='PC%{x}<br>Explica %{y:.1f}% por sí solo<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=x_pc, y=cumulative * 100, name='Varianza acumulada',
    mode='lines+markers', line=dict(color=SEQUENTIAL_BLUE[6], width=2),
    marker=dict(size=8, line=dict(color=INK['surface'], width=2)),
    hovertemplate='PC1 a PC%{x}<br>Acumulan %{y:.1f}%<extra></extra>',
))
fig.add_hline(y=80, line=dict(color=INK['axis'], width=1, dash='dash'))
fig.add_vline(x=n_components, line=dict(color=INK['axis'], width=1, dash='dash'))
fig.add_annotation(x=len(explained), y=80, text='80% de la varianza', showarrow=False,
                    font=dict(size=10.5, color=INK['muted']), xanchor='right', yanchor='bottom')
fig.update_layout(
    title=dict(text='Cuánta información guarda cada componente',
               subtitle=dict(text=f'Con {n_components} componentes se conserva el '
                                   f'{cumulative[n_components-1]:.0%} de la varianza de las 19 métricas')),
    xaxis=dict(title='Componente principal', dtick=1),
    yaxis=dict(title='Varianza explicada (%)'),
    legend=dict(orientation='h', y=-0.18, x=0),
)
render_plot(fig, width=720, height=460)

## 3. Biplot de PCA

El biplot de PC1 vs. PC2 muestra en el mismo gráfico los equipos (puntos) y las
flechas de carga de cada métrica original: la dirección de una flecha es hacia
dónde empuja esa métrica, y su longitud es cuánto pesa en ese plano. Esto es el
**vocabulario de estilo** que se reusa para interpretar los clusters del paso 5.

Los puntos van todos del mismo color a propósito: acá todavía no hay clusters, y
el color en este notebook está reservado para el grupo de estilo. Pasa el mouse
por encima para ver el equipo y su liga. Las flechas van con el nombre
abreviado de cada métrica; la tabla de abajo trae los nombres completos y el
valor exacto de cada carga.

In [7]:
hover_teams = np.stack([df['Squad'], df['liga'].astype(str)], axis=-1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=scores[:, 0], y=scores[:, 1], mode='markers',
    marker=dict(size=9, color=SEQUENTIAL_BLUE[4], opacity=0.82,
                line=dict(color=INK['surface'], width=1.5)),
    customdata=hover_teams,
    hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}'
                  '<br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>',
    showlegend=False,
))

# flechas de carga: cola en el origen, punta en la carga escalada al rango de
# los scores (el factor es solo para que flechas y nube compartan escala visual)
arrow_scale = (np.abs(scores[:, :2]).max() / np.abs(loadings[:, :2]).max()) * 0.82
tips = loadings[:, :2] * arrow_scale
angles = np.arctan2(tips[:, 1], tips[:, 0])

# Etiquetas abreviadas SOLO para el biplot: con los nombres largos de FEATURES
# el texto se pisa entre sí en el abanico de métricas ofensivas (que apuntan
# casi todas en la misma dirección, justamente porque están correlacionadas) y
# además estira los ejes hasta achicar la nube de puntos. Los nombres completos
# están en la tabla de cargas de la celda siguiente.
SHORT = ['Posesión', 'Gls/90', 'Ast/90', 'Sh/90', 'SoT/90', 'SoT%', 'G/Sh', 'G/SoT',
         'Amarillas', 'Rojas', 'Faltas', 'Faltas rec.', 'Fuera juego', 'Centros',
         'Intercep.', 'Tackles', 'GA/90', 'Save%', 'SoTA/90']

# Las métricas ofensivas apuntan casi todas en la misma dirección (están muy
# correlacionadas — Gls/90 y Ast/90 llegan a r≈0.98), así que sus etiquetas se
# encimarían. Separarlas empujándolas por el radio no alcanza (comparten
# dirección Y magnitud) y encima estira los ejes hasta achicar la nube. El
# choque pasa en pantalla, así que se resuelve en pantalla: se fijan los rangos
# de los ejes, se proyecta cada etiqueta a píxeles y, recorriéndolas de flecha
# más larga a más corta, se baja la que pisaría a una ya colocada.
PLOT_W, PLOT_H = 670, 550   # área de dibujo = tamaño de render_plot menos márgenes
LABEL_GAP, LINE_H, CHAR_W = 15, 14, 5.6

pts = np.vstack([scores[:, :2], tips])
span = pts.max(axis=0) - pts.min(axis=0)
x_range = [pts[:, 0].min() - span[0] * 0.10, pts[:, 0].max() + span[0] * 0.14]
y_range = [pts[:, 1].min() - span[1] * 0.08, pts[:, 1].max() + span[1] * 0.08]


def to_px(x, y):
    return ((x - x_range[0]) / (x_range[1] - x_range[0]) * PLOT_W,
            (1 - (y - y_range[0]) / (y_range[1] - y_range[0])) * PLOT_H)


shift, placed = np.zeros(len(feature_cols), dtype=int), []
for i in np.argsort(-np.hypot(tips[:, 0], tips[:, 1])):
    dx, dy = np.cos(angles[i]), np.sin(angles[i])
    px, py = to_px(tips[i, 0] + dx * 0.4, tips[i, 1] + dy * 0.4)
    w = len(SHORT[i]) * CHAR_W + 4
    x0 = px if dx > 0.25 else px - w if dx < -0.25 else px - w / 2
    down = 0
    while down <= 90:
        box = (x0, py + down - LINE_H / 2, x0 + w, py + down + LINE_H / 2)
        if not any(box[0] < b[2] and b[0] < box[2] and box[1] < b[3] and b[1] < box[3]
                   for b in placed):
            break
        down += LABEL_GAP
    placed.append((x0, py + down - LINE_H / 2, x0 + w, py + down + LINE_H / 2))
    shift[i] = -down   # yshift positivo sube en Plotly; bajar en pantalla es negativo

annotations = []
for i, short in enumerate(SHORT):
    fx, fy = tips[i]
    dx, dy = np.cos(angles[i]), np.sin(angles[i])
    annotations.append(dict(
        x=fx, y=fy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.1, arrowwidth=1.2,
        arrowcolor=INK['secondary'], opacity=0.85,
    ))
    annotations.append(dict(
        x=fx + dx * 0.4, y=fy + dy * 0.4, xref='x', yref='y', yshift=int(shift[i]),
        showarrow=False, text=short, font=dict(size=9.5, color=INK['secondary']),
        xanchor='left' if dx > 0.25 else 'right' if dx < -0.25 else 'center',
        yanchor='middle',
    ))

fig.add_hline(y=0, line=dict(color=INK['grid'], width=1))
fig.add_vline(x=0, line=dict(color=INK['grid'], width=1))
fig.update_layout(
    title=dict(text='Biplot de PCA — el vocabulario de estilo',
               subtitle=dict(text='Cada flecha es una métrica: hacia dónde empuja y cuánto pesa · '
                                   'pasa el mouse sobre un punto para ver el equipo')),
    xaxis=dict(title=f'PC1 ({pc1_var:.1%} de la varianza)', range=x_range),
    yaxis=dict(title=f'PC2 ({pc2_var:.1%} de la varianza)', range=y_range),
    annotations=annotations,
)
render_plot(fig, width=780, height=700)

In [8]:
loadings_df = pd.DataFrame(loadings[:, :2], index=[FEATURES[f] for f in feature_cols],
                            columns=['PC1', 'PC2'])
for pc in ['PC1', 'PC2']:
    orden = loadings_df[pc].abs().sort_values(ascending=False).index
    print(f'--- {pc}: mayores cargas (por valor absoluto) ---')
    print(loadings_df[pc].reindex(orden).round(2).head(6).to_string())
    print()

--- PC1: mayores cargas (por valor absoluto) ---
Goles/90             0.96
Asistencias/90       0.95
Tiros a puerta/90    0.91
Posesión (%)         0.85
Tiros/90             0.85
Goles por tiro       0.84

--- PC2: mayores cargas (por valor absoluto) ---
Atajadas (%)                   0.71
Goles recibidos/90            -0.70
Faltas recibidas/90            0.68
Intercepciones/90             -0.53
Tiros a puerta recibidos/90   -0.41
Faltas cometidas/90            0.30



### Lectura de los ejes

**PC1 — dominio ofensivo** (~38% de la varianza): goles/90, asistencias/90,
tiros a puerta/90, posesión y goles por tiro cargan todas juntas y con el mismo
signo. No es un eje de "estilo" en el sentido de *cómo* ataca un equipo, sino de
*cuánto* domina — posesión, generación y conversión de tiro van pegadas entre
los 96 equipos, así que el eje que más varianza explica termina siendo, en la
práctica, un proxy de nivel ofensivo.

**PC2 — solidez defensiva** (~13% de la varianza): atajadas (%) y faltas
recibidas empujan hacia un lado; goles recibidos/90, intercepciones/90 y tiros a
puerta recibidos/90 hacia el otro. Este sí es un eje más propiamente defensivo:
separa equipos que conceden poco y resuelven bien en portería (sin necesitar
recuperar mucho el balón, quizás porque el rival ya no llega) de los que
enfrentan más volumen y se apoyan en el portero o en intercepciones para
compensar.

Este vocabulario (PC1 = cuánto domina, PC2 = qué tan sólido defiende) es el que
se usa abajo para nombrar los clusters.

## 4. Gráfico de codo + silhouette score

Para elegir *k* de forma justificada se cruzan dos criterios sobre los
`n_components` de PCA (no solo PC1/PC2 del biplot):

- **Inercia (codo)** — dónde agregar un cluster más deja de reducir mucho la
  varianza intra-cluster. Lo que importa no es la curva en sí sino **cuánto
  baja de un k al siguiente**: el codo es donde esa ganancia marginal se cae.
- **Silhouette score** — qué tan bien separados y compactos quedan los clusters
  (más alto = mejor), que compensa la subjetividad de "dónde está el codo".

In [9]:
k_range = list(range(2, 11))
inertias, silhouettes = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RNG, n_init=10).fit(scores)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(scores, km.labels_))

# ganancia marginal: cuánto baja la inercia respecto del k anterior
ganancia = [np.nan] + [(inertias[i - 1] - inertias[i]) / inertias[i - 1] * 100
                        for i in range(1, len(inertias))]

pd.DataFrame({
    'k': k_range,
    'Inercia': np.round(inertias, 1),
    'Ganancia vs. k-1 (%)': np.round(ganancia, 1),
    'Silhouette': np.round(silhouettes, 3),
})

,k,Inercia,Ganancia vs. k-1 (%),Silhouette
0,2,1076.0,NaN,0.258
1,3,946.2,12.1,0.162
2,4,856.5,9.5,0.143
3,5,784.6,8.4,0.148
4,6,714.8,8.9,0.161
5,7,681.0,4.7,0.137
6,8,650.2,4.5,0.149
7,9,615.8,5.3,0.115
8,10,577.1,6.3,0.129


In [10]:
# sin subplot_titles: chocaban con el subtítulo del gráfico y además repetían
# lo que ya dice el título de cada eje Y
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13)

fig.add_trace(go.Scatter(
    x=k_range, y=inertias, mode='lines+markers', name='Inercia',
    line=dict(color=SEQUENTIAL_BLUE[5], width=2),
    marker=dict(size=8, line=dict(color=INK['surface'], width=2)),
    customdata=np.round(ganancia, 1),
    hovertemplate='k=%{x}<br>Inercia: %{y:.0f}<br>Baja %{customdata:.1f}% vs. k-1<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=k_range, y=silhouettes, mode='lines+markers', name='Silhouette',
    line=dict(color=SEQUENTIAL_BLUE[5], width=2),
    marker=dict(size=8, line=dict(color=INK['surface'], width=2)),
    hovertemplate='k=%{x}<br>Silhouette: %{y:.3f}<extra></extra>',
), row=1, col=2)

for col in (1, 2):
    fig.add_vline(x=6, line=dict(color=INK['axis'], width=1, dash='dash'), row=1, col=col)
    fig.update_xaxes(title_text='k (número de clusters)', dtick=1, row=1, col=col)
fig.update_yaxes(title_text='Inercia (WCSS)', row=1, col=1)
fig.update_yaxes(title_text='Silhouette score', row=1, col=2)
fig.update_layout(
    title=dict(text='Cómo elegimos k',
               subtitle=dict(text='La línea marca k=6 · pasa el mouse por la inercia para ver '
                                   'cuánto baja respecto del k anterior')),
    showlegend=False,
    margin=dict(t=135),
)
render_plot(fig, width=860, height=440)

### Por qué k=6

Los dos criterios apuntan a lados distintos, así que hay que leerlos juntos en
vez de quedarse con el número más alto:

- **El codo apoya k=6.** Cada cluster que se agrega hasta el sexto baja la
  inercia entre 8% y 12%; el séptimo baja **4.7%**, casi la mitad. Ahí es
  exactamente donde la curva deja de pagar, que es la definición del codo.
- **El silhouette máximo está en k=2 (0.258), no en k=6 (0.161)** — hay que
  decirlo tal cual. Pero k=2 parte a los 96 equipos en "domina mucho" vs.
  "domina poco", que es básicamente el signo de PC1: una división real pero
  trivial, y muy cercana a ordenar la tabla por nivel, que es justo lo que se
  intentó dejar fuera al excluir `PPM` y `+/-` de las features.
- Entre las particiones más finas, **k=6 es un máximo local** de silhouette
  (0.161, prácticamente empatado con el 0.162 de k=3 y por encima de k=4, k=5 y
  k=7), así que no se está eligiendo un k arbitrario dentro de esa zona.
Ningún cluster queda vacío ni convertido en un caso aislado: el más chico tiene
6 equipos y el más grande 27.

> **Un argumento que NO vale, aunque tiente.** La red del paso 7 también
> encuentra 6 comunidades, y es muy tentador presentarlo como confirmación
> independiente. No lo es: ese número depende del `K_NEIGHBORS` que elijamos al
> construir el grafo, y barriéndolo de 3 a 10 da 9, 7, 6, 7, 5 y 3 comunidades
> (se verifica en la sección 7). Encuentra 6 *porque* elegimos 5 vecinos. La
> justificación de k=6 se apoya solo en el codo y en el máximo local de
> silhouette.

In [11]:
K = 6
kmeans = KMeans(n_clusters=K, random_state=RNG, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(scores)

print('Tamaño de cada cluster:')
print(df['cluster_kmeans'].value_counts().sort_index().to_string())
print(f'\nSilhouette con k={K}: {silhouette_score(scores, df["cluster_kmeans"]):.3f}')

Tamaño de cada cluster:
cluster_kmeans
0    15
1    18
2    27
3    13
4     6
5    17

Silhouette con k=6: 0.161


### ¿Y si no hay clusters? El test que casi nadie corre

Un silhouette de 0.16 es bajo, pero "bajo" comparado con qué. La única forma de
saberlo es comparar contra un **nulo**: datos generados con la misma matriz de
covarianza que los reales pero **sin ninguna estructura de grupos por
construcción** (una nube gaussiana). Si k-means saca el mismo silhouette de una
nube sin clusters que de nuestros datos, entonces lo que encontramos no son
grupos: es un corte arbitrario de un continuo.

In [12]:
rng = np.random.default_rng(0)
cov = np.cov(X, rowvar=False)

filas = []
for k in [2, 4, 6, 8]:
    real = silhouette_score(scores, KMeans(n_clusters=k, random_state=RNG, n_init=10).fit_predict(scores))
    sims = []
    for _ in range(30):
        X_nulo = rng.multivariate_normal(np.zeros(len(feature_cols)), cov, size=len(df))
        s_nulo = PCA(n_components=n_components, random_state=RNG).fit_transform(
            StandardScaler().fit_transform(X_nulo))
        sims.append(silhouette_score(s_nulo, KMeans(n_clusters=k, random_state=RNG,
                                                    n_init=10).fit_predict(s_nulo)))
    filas.append({'k': k, 'Silhouette real': round(real, 3),
                  'Nulo (media)': round(np.mean(sims), 3),
                  'Nulo (desv.)': round(np.std(sims), 3),
                  'z': round((real - np.mean(sims)) / np.std(sims), 1)})

pd.DataFrame(filas).set_index('k')

,Silhouette real,Nulo (media),Nulo (desv.),z
k,,,,
2,0.258,0.247,0.023,0.5
4,0.143,0.162,0.016,-1.2
6,0.161,0.152,0.012,0.8
8,0.149,0.144,0.015,0.3


**No hay estructura de clusters.** El silhouette real es indistinguible del
que sale de una nube sin grupos: todos los z quedan dentro de ±1.5, cuando haría
falta un z alto y positivo para hablar de grupos reales.

Esto no invalida lo que sigue, pero **cambia qué significa**. Los 6 grupos no son
tribus que estaban ahí esperando ser descubiertas: son una **segmentación** de
una nube continua, como los rangos de edad o los tramos de un impuesto. Útil para
describir y comparar, siempre que no se afirme que existen fronteras naturales
entre ellos. De acá en adelante se los llama arquetipos por comodidad, pero con
esa aclaración puesta.

## 5. K-means + análisis manual

Clusters duros sobre los `n_components` de PCA, interpretados con el vocabulario
del biplot.

### Perfil de cada cluster

El mapa de calor muestra el perfil en **z-scores**: cuántas desviaciones
estándar está cada cluster por encima (azul) o por debajo (rojo) del promedio de
los 96 equipos en cada métrica. Es más legible que una tabla de promedios
crudos, porque pone las 19 métricas en la misma escala y deja ver de un vistazo
qué define a cada grupo.

In [13]:
# X ya está estandarizado, así que el promedio de un cluster en X ES su z-score
profile_z = X.groupby(df['cluster_kmeans'].values).mean()
profile_z.index = [f'Cluster {c}' for c in profile_z.index]

fig = go.Figure(go.Heatmap(
    z=profile_z.T.values,
    x=profile_z.index,
    y=[FEATURES[f] for f in feature_cols],
    colorscale=DIVERGING_SCALE, zmid=0,
    xgap=2, ygap=2,
    colorbar=dict(title=dict(text='z-score', side='right'), thickness=14, outlinewidth=0),
    hovertemplate='%{x}<br>%{y}<br>%{z:+.2f} desviaciones vs. el promedio<extra></extra>',
))
fig.update_layout(
    title=dict(text='Qué define a cada arquetipo',
               subtitle=dict(text='Azul = por encima del promedio de los 96 equipos, rojo = por debajo')),
    xaxis=dict(side='top', title=None, showgrid=False),
    yaxis=dict(autorange='reversed', title=None, showgrid=False),
    margin=dict(l=200, t=155),
)
render_plot(fig, width=760, height=680)

In [14]:
# promedios en unidades originales, para leer los números reales
perfil = df.groupby('cluster_kmeans', observed=True)[feature_cols].mean().T
perfil.index = [FEATURES[f] for f in perfil.index]
perfil.columns = [f'Cluster {c}' for c in perfil.columns]
perfil.round(2)

,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5
Posesión (%),56.73,52.31,45.69,45.08,62.73,47.98
Goles/90,1.66,1.61,1.18,0.90,2.35,1.07
Asistencias/90,1.20,1.13,0.78,0.61,1.77,0.73
Tiros/90,14.32,13.41,11.57,11.27,17.28,11.14
Tiros a puerta/90,4.94,4.80,3.84,3.63,6.78,3.70
Precisión de tiro (%),34.60,35.84,33.22,32.10,39.15,33.32
Goles por tiro,0.11,0.11,0.09,0.07,0.13,0.09
Goles por tiro a puerta,0.31,0.31,0.28,0.22,0.32,0.26
Amarillas/90,1.68,2.14,1.84,2.31,1.60,2.05
Rojas/90,0.06,0.11,0.07,0.16,0.07,0.13


In [15]:
for c in sorted(df['cluster_kmeans'].unique()):
    equipos = df.loc[df['cluster_kmeans'] == c, 'Squad'].tolist()
    print(f'--- Cluster {c} ({len(equipos)} equipos) ---')
    print(', '.join(equipos))
    print()

--- Cluster 0 (15 equipos) ---
Dortmund, Leverkusen, Atalanta, Como, Inter, Juventus, Milan, Napoli, Roma, Lyon, Arsenal, Aston Villa, Liverpool, Manchester City, Newcastle

--- Cluster 1 (18 equipos) ---
Frankfurt, Hoffenheim, Lens, Lille, Marseille, Monaco, Rennes, Strasbourg, Toulouse, Atlético Madrid, Real Betis, Real Sociedad, Villarreal, Bournemouth, Brighton, Chelsea, Manchester Utd, Tottenham

--- Cluster 2 (27 equipos) ---
Augsburg, Freiburg, Gladbach, Heidenheim, Köln, Mainz 05, St Pauli, Union Berlin, Werder Bremen, Wolfsburg, Torino, Udinese, Angers, Brest, Lorient, Metz, Nice, Paris FC, Valencia, Brentford, Burnley, Crystal Palace, Everton, Fulham, Leeds United, Nottingham, West Ham

--- Cluster 3 (13 equipos) ---
Hellas Verona, Pisa, Auxerre, Le Havre, Nantes, Alavés, Athletic Club, Espanyol, Getafe, Oviedo, Rayo Vallecano, Sevilla, Wolves

--- Cluster 4 (6 equipos) ---
Bayern Munich, RB Leipzig, Stuttgart, PSG, Barcelona, Real Madrid

--- Cluster 5 (17 equipos) ---
Hambu

### Los seis arquetipos

Leyendo el mapa de calor y los promedios con el vocabulario del biplot (PC1 =
cuánto domina, PC2 = qué tan sólido defiende):

| Cluster | Nombre | Qué lo define | Ejemplos |
|---|---|---|---|
| **4** (6) | **Aplanadoras** | El extremo de PC1: 62.7% de posesión, 17.3 tiros y 2.35 goles por partido, la mejor precisión de tiro (39%) y **las menos faltas de todos** (9.4) — no necesitan interrumpir porque casi no defienden. | Bayern, PSG, Real Madrid, Barcelona |
| **0** (15) | **Control con portería sólida** | Alto en los dos ejes: 56.7% de posesión y, sobre todo, **la mejor defensa del estudio** (1.04 goles recibidos, 71.1% de atajadas, los menos tiros a puerta en contra) con muy pocas amarillas (1.68) y **las menos intercepciones** (7.1) — recuperan poco porque el rival llega poco. | Inter, Liverpool, Napoli, Arsenal |
| **1** (18) | **Ofensivos de presión** | Ataque casi idéntico al del cluster 0 (52% posesión, 1.61 goles) pero defendiendo al revés: **recuperan mucho más** (10.2 tackles y 8.8 intercepciones, contra 8.9 y 7.1 del cluster 0) y pagan con más amarillas (2.14). Roban arriba en vez de esperar. | Atlético, Chelsea, Marsella, Brighton |
| **5** (17) | **Repliegue con portero salvador** | El menor volumen de tiro del estudio (11.1) y poca recuperación activa (8.9 tackles, 7.7 intercepciones), pero **la segunda mejor tasa de atajadas (69.6%)** y **las más faltas recibidas (12.8)**: ceden el balón, frenan al rival con falta y confían en el portero. | Bologna, Fiorentina, Osasuna, Celta |
| **3** (13) | **Fricción y pocas ocasiones** | El arquetipo más áspero: **las más faltas (13.9), amarillas (2.31) y rojas (0.16)** de las cinco ligas, con la posesión (45.1%) y la conversión (0.07 goles por tiro) más bajas. Partido trabado. | Getafe, Rayo, Athletic, Sevilla |
| **2** (27) | **Bloque expuesto** | El grupo más grande y el más castigado atrás: 45.7% de posesión, **la peor defensa (1.64 goles recibidos, 4.7 tiros a puerta en contra, 65% de atajadas)** y sin compensarlo con faltas ni presión. | Augsburg, Everton, Burnley, Werder |

Lo que sí separa *estilo* y no solo nivel son los tres del medio: **0, 1 y 5**
tienen volúmenes ofensivos parecidos pero defienden de tres maneras distintas —
esperar y atajar (0), presionar y robar (1), o replegarse y cortar con falta
(5). Esa es la dimensión que aporta PC2.

Pero conviene leer esta tabla de reojo: si se ordenan los seis grupos por
cualquiera de las columnas ofensivas, salen casi en el mismo orden. Eso no es
casualidad y no es un detalle — es el tema de la sección siguiente.

In [16]:
hover_km = np.stack([df['Squad'], df['liga'].astype(str),
                      df['cluster_kmeans'].astype(str)], axis=-1)

fig = go.Figure()
for c in sorted(df['cluster_kmeans'].unique()):
    m = (df['cluster_kmeans'] == c).values
    fig.add_trace(go.Scatter(
        x=scores[m, 0], y=scores[m, 1], mode='markers', name=f'Cluster {c}',
        marker=dict(size=10, color=CLUSTER_COLORS[c], opacity=0.88,
                    line=dict(color=INK['surface'], width=1.5)),
        customdata=hover_km[m],
        hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}'
                      '<br>Cluster %{customdata[2]}<extra></extra>',
    ))
fig.add_hline(y=0, line=dict(color=INK['grid'], width=1))
fig.add_vline(x=0, line=dict(color=INK['grid'], width=1))
fig.update_layout(
    title=dict(text=f'Los {K} arquetipos en el plano principal',
               subtitle=dict(text='Los clusters se calcularon sobre los 7 componentes; acá se ven '
                                   'proyectados en los 2 primeros · pasa el mouse para ver el equipo')),
    xaxis=dict(title=f'PC1 — dominio ofensivo ({pc1_var:.1%})'),
    yaxis=dict(title=f'PC2 — solidez defensiva ({pc2_var:.1%})'),
    legend=dict(orientation='h', y=-0.16, x=0),
)
render_plot(fig, width=800, height=640)

### ¿Los clusters son en realidad ligas disfrazadas?

La liga nunca entró al modelo, así que vale la pena comprobar si igual reapareció
sola (sería el caso si, por ejemplo, un cluster fuera "la Premier"). Si los
arquetipos cruzan ligas, entonces capturan estilo y no procedencia.

In [17]:
cross = pd.crosstab(df['liga'], df['cluster_kmeans'])
cross.columns = [f'Cluster {c}' for c in cross.columns]
print('Equipos de cada liga en cada arquetipo:')
print(cross.to_string())
print(f'\nNingún cluster es monoliga: cada uno reúne equipos de '
      f'{cross.gt(0).sum(axis=0).min()} a {cross.gt(0).sum(axis=0).max()} ligas distintas.')

Equipos de cada liga en cada arquetipo:
                Cluster 0  Cluster 1  Cluster 2  Cluster 3  Cluster 4  Cluster 5
liga                                                                            
Bundesliga              2          2         10          0          3          1
Serie A                 7          0          2          2          0          9
Ligue 1                 1          7          6          3          1          0
La Liga                 0          4          1          7          2          6
Premier League          5          5          8          1          0          1

Ningún cluster es monoliga: cada uno reúne equipos de 3 a 5 ligas distintas.


## 5.1 La pregunta incómoda: ¿esto es estilo o es la tabla de posiciones?

Los clusters no son ligas disfrazadas. Pero queda una sospecha peor, que ya
asomó al leer PC1: si el eje que más manda mezcla posesión, goles, asistencias y
conversión… eso se parece mucho a **ser bueno**, no a un estilo.

Hay una forma directa de comprobarlo: `PPM` (puntos por partido) quedó **fuera**
del modelo justamente por ser resultado. Si los clusters lo predicen igual, es
que lo reconstruyeron sin permiso.

In [18]:
ppm = df['pt_Team Success_PPM']

print(f'corr(PC1, puntos por partido) = {np.corrcoef(scores[:, 0], ppm)[0, 1]:+.2f}')
print(f'corr(PC2, puntos por partido) = {np.corrcoef(scores[:, 1], ppm)[0, 1]:+.2f}')

# eta²: fracción de la variación en PPM que explican los clusters
pred = df.groupby('cluster_kmeans', observed=True)['pt_Team Success_PPM'].transform('mean')
eta2_ppm = 1 - ((ppm - pred) ** 2).sum() / ((ppm - ppm.mean()) ** 2).sum()
print(f'\nLos clusters explican el {eta2_ppm:.0%} de la variación en puntos por partido')

print('\nPuntos por partido medio de cada arquetipo:')
print(df.groupby('cluster_kmeans', observed=True)['pt_Team Success_PPM']
        .agg(Equipos='size', PPM='mean').round(2).sort_values('PPM', ascending=False).to_string())

corr(PC1, puntos por partido) = +0.93
corr(PC2, puntos por partido) = +0.22

Los clusters explican el 70% de la variación en puntos por partido

Puntos por partido medio de cada arquetipo:
                Equipos   PPM
cluster_kmeans               
4                     6  2.22
0                    15  1.85
1                    18  1.59
5                    17  1.18
2                    27  1.10
3                    13  0.95


In [19]:
fig = go.Figure()
for c in sorted(df['cluster_kmeans'].unique()):
    m = (df['cluster_kmeans'] == c).values
    fig.add_trace(go.Scatter(
        x=scores[m, 0], y=ppm[m], mode='markers', name=f'Cluster {c}',
        marker=dict(size=10, color=CLUSTER_COLORS[c], opacity=0.88,
                    line=dict(color=INK['surface'], width=1.5)),
        customdata=hover_km[m],
        hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}'
                      '<br>PC1: %{x:.2f}<br>%{y:.2f} puntos por partido<extra></extra>',
    ))
r = np.corrcoef(scores[:, 0], ppm)[0, 1]
fit = np.poly1d(np.polyfit(scores[:, 0], ppm, 1))
xs = np.array([scores[:, 0].min(), scores[:, 0].max()])
fig.add_trace(go.Scatter(x=xs, y=fit(xs), mode='lines', name=f'r = {r:.2f}',
                          line=dict(color=INK['muted'], width=2, dash='dash'),
                          hoverinfo='skip'))
fig.update_layout(
    title=dict(text='PC1 no es un estilo: es la tabla de posiciones',
               subtitle=dict(text='Los puntos por partido nunca entraron al modelo, y aun así '
                                   'el primer componente los reconstruye casi perfecto')),
    xaxis=dict(title=f'PC1 — supuesto "dominio ofensivo" ({pc1_var:.1%} de la varianza)'),
    yaxis=dict(title='Puntos por partido (fuera del modelo)'),
    legend=dict(orientation='h', y=-0.16, x=0),
)
render_plot(fig, width=800, height=560)

### El hallazgo principal del notebook

`r = +0.93`. Los clusters explican el **70%** de la variación en puntos por
partido, y ordenados por PPM medio quedan casi como una tabla de posiciones. El
eje que domina el 46% de la distancia con la que k-means agrupa **no mide cómo
juega un equipo: mide qué tan bueno es**.

¿De dónde salió, si excluimos `PPM` y `+/-` justamente por ser resultado? De la
grieta anunciada en la sección 1: dejamos dentro `Goles/90` y `Goles
recibidos/90`, que son las dos componentes de la diferencia de gol. Excluimos la
etiqueta y conservamos los ingredientes.

Y hay una segunda causa, más sutil: **clusterizamos sobre los componentes sin
blanquear**. Los scores de PCA conservan la escala de cada componente, y PC1
tiene varianza 7.3 contra 0.9 del PC7 — es decir, PC1 pesa unas ocho veces más
que el último componente al calcular distancias. Aunque las features fueran
perfectas, la geometría ya estaba inclinada hacia el eje dominante.

Nada de esto invalida las secciones 6, 7 y 8: los métodos hacen bien lo suyo
sobre el espacio que les dimos. Lo que queda tocado es la **interpretación**.
Con eso claro, la sección 9 rehace el ejercicio sin nivel y responde la pregunta
que de verdad importa: *quitando lo bueno que es cada equipo, ¿queda estilo?*

## 6. GMM

Mismo *k*, mismos componentes de PCA, pero con Gaussian Mixture Models: cada
equipo tiene una probabilidad de pertenecer a cada cluster en vez de una
asignación dura. Esto deja ver quién es "100% un estilo" y quién vive a mitad de
camino entre dos.

### Antes de correrlo: dos ajustes sin los cuales el GMM no sirve

**1. El tipo de covarianza.** El default de scikit-learn es
`covariance_type='full'`, que le da a cada componente una matriz completa. Con
6 componentes en 7 dimensiones son **215 parámetros libres para 96 equipos** —
más parámetros que datos. El modelo se sobreajusta hasta volverse determinista:
asigna casi todos los equipos con más del 90% de probabilidad y **no deja
prácticamente ningún híbrido**, que es justo lo único que GMM aporta sobre
K-means. En vez de elegir a mano una covarianza más simple, se compara por
**BIC**, que penaliza la verosimilitud por número de parámetros.

**2. Un piso para la varianza (`reg_covar`).** Corriendo el GMM con el default
(`reg_covar=1e-6`), un componente colapsa sobre un solo equipo con varianza
**exactamente 0**. Esto es la singularidad clásica de los modelos de mezcla: la
verosimilitud tiende a infinito cuando la varianza de un componente tiende a
cero, así que **el BIC es inservible justo en esa solución** — la premia por ser
degenerada, no por ser buena. Se le pone un piso a la varianza del 5% de la
varianza media de los componentes de PCA, y recién ahí se comparan los tipos.

In [20]:
reg_covar = 0.05 * scores.var(axis=0).mean()
print(f'Piso de varianza: {reg_covar:.3f} '
      f'(5% de la varianza media de los {n_components} componentes)\n')

bic_scores = {}
for cv in ['full', 'tied', 'diag', 'spherical']:
    g = GaussianMixture(n_components=K, covariance_type=cv, reg_covar=reg_covar,
                        random_state=RNG, n_init=10).fit(scores)
    p_max = g.predict_proba(scores).max(axis=1)
    bic_scores[cv] = {'BIC': round(g.bic(scores), 1),
                      'Parámetros': int(g._n_parameters()),
                      'Híbridos (<60%)': int((p_max < 0.60).sum()),
                      'Puros (>90%)': int((p_max > 0.90).sum())}

cov_type = min(bic_scores, key=lambda c: bic_scores[c]['BIC'])
print(f'BIC elige covariance_type=\'{cov_type}\' (menor BIC = mejor)\n')
pd.DataFrame(bic_scores).T

Piso de varianza: 0.114 (5% de la varianza media de los 7 componentes)



BIC elige covariance_type='spherical' (menor BIC = mejor)



,BIC,Parámetros,Híbridos (<60%),Puros (>90%)
full,2943.6,215.0,1.0,84.0
tied,2511.3,75.0,6.0,67.0
diag,2557.8,89.0,9.0,55.0
spherical,2458.0,53.0,8.0,48.0


In [21]:
gmm = GaussianMixture(n_components=K, covariance_type=cov_type, reg_covar=reg_covar,
                      random_state=RNG, n_init=10)
df['cluster_gmm'] = gmm.fit_predict(scores)
gmm_proba = gmm.predict_proba(scores)
df['gmm_proba_max'] = gmm_proba.max(axis=1)

HYBRID_THRESHOLD = 0.60
df['gmm_hibrido'] = df['gmm_proba_max'] < HYBRID_THRESHOLD

print(f'{df["gmm_hibrido"].sum()} de {len(df)} equipos son híbridos '
      f'(su cluster más probable no llega al {HYBRID_THRESHOLD:.0%})')
print(f'Equipos "puros" (>90% en un solo cluster): {(df["gmm_proba_max"] > 0.9).sum()}')
print('\nTamaño de cada componente:')
print(df['cluster_gmm'].value_counts().sort_index().to_string())

8 de 96 equipos son híbridos (su cluster más probable no llega al 60%)
Equipos "puros" (>90% en un solo cluster): 48

Tamaño de cada componente:
cluster_gmm
0    14
1    22
2    16
3     1
4    29
5    14


In [22]:
# los 15 equipos más ambiguos: cómo se reparte su probabilidad entre clusters
ambiguos = df.nsmallest(15, 'gmm_proba_max')
orden = ambiguos.index[::-1]  # el más híbrido arriba

fig = go.Figure()
for c in range(K):
    fig.add_trace(go.Bar(
        y=df.loc[orden, 'Squad'], x=gmm_proba[orden, c] * 100,
        name=f'Cluster {c}', orientation='h',
        marker=dict(color=CLUSTER_COLORS[c], line=dict(color=INK['surface'], width=1)),
        hovertemplate='<b>%{y}</b><br>Cluster ' + str(c) + ': %{x:.0f}%<extra></extra>',
    ))
fig.update_layout(
    barmode='stack',
    title=dict(text='Los equipos más híbridos',
               subtitle=dict(text='Cómo reparte GMM la pertenencia de los 15 equipos menos '
                                   'definidos · una barra de un solo color sería un estilo puro')),
    xaxis=dict(title='Probabilidad de pertenencia (%)', range=[0, 100]),
    yaxis=dict(title=None),
    legend=dict(orientation='h', y=-0.13, x=0),
    margin=dict(l=150),
)
render_plot(fig, width=820, height=620)

In [23]:
df.loc[df['gmm_hibrido'], ['Squad', 'liga', 'cluster_kmeans', 'cluster_gmm', 'gmm_proba_max']] \
  .sort_values('gmm_proba_max').round(2)

,Squad,liga,cluster_kmeans,cluster_gmm,gmm_proba_max
55,Toulouse,Ligue 1,1,2,0.47
18,Atalanta,Serie A,0,5,0.48
90,Newcastle,Premier League,0,5,0.50
61,Elche,La Liga,5,5,0.52
70,Real Betis,La Liga,1,5,0.53
38,Angers,Ligue 1,2,4,0.53
66,Mallorca,La Liga,5,5,0.56
92,Sunderland,Premier League,5,4,0.59


**Un componente para un solo equipo.** Aun con el piso de varianza, el GMM
gasta uno de sus seis componentes en **Bayern Munich y nadie más**. No es un
problema numérico —el piso ya descartó esa explicación—, es una propiedad de los
datos: Bayern está tan lejos del resto (se ve en el biplot, es el punto suelto a
la derecha del todo) que al modelo le conviene darle una gaussiana propia antes
que meterlo con nadie. K-means no tiene esa libertad: al minimizar distancias
está obligado a repartir los 96 equipos entre 6 grupos, así que mete a Bayern
con PSG, Madrid y Barcelona. Las dos lecturas son defendibles y la diferencia
entre ellas es, justamente, lo que cada método asume.

## 7. Redes de similitud + community detection

Grafo de equipos: un nodo por equipo, una arista si dos equipos son
suficientemente similares en estilo (distancia euclidiana pequeña sobre los
componentes de PCA). Se corre detección de comunidades (modularidad greedy)
sobre ese grafo — a diferencia de K-means/GMM, no asume que los grupos tengan
forma esférica en el espacio de features **ni exige decir cuántos grupos hay de
antemano**, así que es una tercera mirada genuinamente distinta.

In [24]:
from scipy.spatial.distance import pdist, squareform
from networkx.algorithms.community import greedy_modularity_communities

dist_matrix = squareform(pdist(scores, metric='euclidean'))
similarity = 1 / (1 + dist_matrix)
np.fill_diagonal(similarity, 0)

# grafo k-NN: cada equipo se conecta con sus K_NEIGHBORS más parecidos (no un
# umbral fijo de distancia) para que la densidad del grafo no dependa de la
# escala arbitraria de los componentes de PCA
K_NEIGHBORS = 5
squads = df['Squad'].tolist()
G = nx.Graph()
G.add_nodes_from(squads)
for i, squad in enumerate(squads):
    for j in np.argsort(-similarity[i])[:K_NEIGHBORS]:
        G.add_edge(squad, squads[j], weight=similarity[i, j])

communities = greedy_modularity_communities(G, weight='weight')
squad_to_community = {s: idx for idx, comm in enumerate(communities) for s in comm}
df['comunidad'] = df['Squad'].map(squad_to_community)

print(f'{len(communities)} comunidades detectadas sobre {G.number_of_nodes()} nodos y '
      f'{G.number_of_edges()} aristas (el método eligió solo cuántas hay)')
for idx, comm in enumerate(communities):
    print(f'\nComunidad {idx} ({len(comm)} equipos): {", ".join(sorted(comm))}')

6 comunidades detectadas sobre 96 nodos y 351 aristas (el método eligió solo cuántas hay)



Comunidad 0 (23 equipos): Auxerre, Bologna, Cagliari, Crystal Palace, Elche, Everton, Fiorentina, Fulham, Genoa, Lecce, Leeds United, Levante, Nantes, Newcastle, Nottingham, Parma, Sunderland, Torino, Tottenham, Toulouse, Udinese, Union Berlin, Valencia

Comunidad 1 (18 equipos): Arsenal, Atalanta, Barcelona, Bayern Munich, Como, Dortmund, Inter, Juventus, Leverkusen, Liverpool, Manchester City, Milan, Napoli, PSG, RB Leipzig, Real Madrid, Roma, Stuttgart

Comunidad 2 (16 equipos): Atlético Madrid, Augsburg, Bournemouth, Brighton, Chelsea, Hoffenheim, Lens, Lille, Lyon, Mainz 05, Manchester Utd, Marseille, Monaco, Paris FC, Real Sociedad, Rennes

Comunidad 3 (14 equipos): Alavés, Athletic Club, Cremonese, Espanyol, Getafe, Girona, Hamburger SV, Lazio, Le Havre, Mallorca, Osasuna, Oviedo, Rayo Vallecano, Sevilla

Comunidad 4 (13 equipos): Angers, Brest, Burnley, Gladbach, Hellas Verona, Metz, Nice, Pisa, St Pauli, Werder Bremen, West Ham, Wolfsburg, Wolves

Comunidad 5 (12 equipos): A

### Antes de festejar: ¿6 comunidades, o 6 porque elegimos 5 vecinos?

Que la red encuentre 6 comunidades sin que se lo pidamos es el tipo de
coincidencia que da ganas de usarla como confirmación de k=6. Pero `K_NEIGHBORS`
es un parámetro que elegimos nosotros, así que hay que ver cuánto depende de él
antes de sacarle conclusiones.

In [25]:
for kn in [3, 4, 5, 6, 8, 10]:
    g = nx.Graph()
    g.add_nodes_from(squads)
    for i, squad in enumerate(squads):
        for j in np.argsort(-similarity[i])[:kn]:
            g.add_edge(squad, squads[j], weight=similarity[i, j])
    marca = '   <-- el que usamos' if kn == K_NEIGHBORS else ''
    print(f'K_NEIGHBORS={kn:2d} -> {len(greedy_modularity_communities(g, weight="weight")):2d} '
          f'comunidades{marca}')

K_NEIGHBORS= 3 ->  9 comunidades
K_NEIGHBORS= 4 ->  7 comunidades
K_NEIGHBORS= 5 ->  6 comunidades   <-- el que usamos
K_NEIGHBORS= 6 ->  7 comunidades


K_NEIGHBORS= 8 ->  5 comunidades
K_NEIGHBORS=10 ->  3 comunidades


**El número de comunidades no es un hallazgo, es un eco del parámetro**: va de
9 a 3 según cuántos vecinos conectemos. Que dé 6 con `K_NEIGHBORS=5` es una
coincidencia, no una validación de k=6 — por eso el argumento se retiró de la
sección 4. Lo que sí aporta la red, y no depende tanto del parámetro, es la
*forma* de los vecindarios: quién queda cerca de quién.

In [26]:
# iterations=300 (el default son 50): con 96 nodos y 351 aristas el layout no
# alcanza a converger en 50 pasos y queda una "bola de pelos" donde no se
# distingue ninguna estructura, aunque el grafo sí la tenga
pos = nx.spring_layout(G, weight='weight', seed=RNG, k=0.6, iterations=300)

edge_x, edge_y = [], []
for u, v in G.edges():
    edge_x += [pos[u][0], pos[v][0], None]
    edge_y += [pos[u][1], pos[v][1], None]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                         line=dict(color=INK['grid'], width=1),
                         hoverinfo='skip', showlegend=False))

# En ESTE gráfico el color es la comunidad (el agrupamiento que encontró la
# propia red), no el cluster de K-means: es el resultado del método que estamos
# mirando. El cruce entre ambos va en el mapa de calor de abajo, medido.
for idx in range(len(communities)):
    nodes = sorted(communities[idx])
    fig.add_trace(go.Scatter(
        x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes],
        mode='markers', name=f'Comunidad {idx}',
        marker=dict(size=11, color=CLUSTER_COLORS[idx], opacity=0.9,
                    line=dict(color=INK['surface'], width=1.5)),
        customdata=[[n, str(df.loc[df['Squad'] == n, 'liga'].iloc[0]),
                     int(df.loc[df['Squad'] == n, 'cluster_kmeans'].iloc[0])] for n in nodes],
        hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}'
                      f'<br>Comunidad {idx} · cluster K-means %{{customdata[2]}}<extra></extra>',
    ))

fig.update_layout(
    title=dict(text='Red de similitud de estilo',
               subtitle=dict(text=f'Cada equipo unido a sus {K_NEIGHBORS} más parecidos · '
                                   'la posición la da el grafo, el color la comunidad detectada'),),
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    legend=dict(orientation='h', y=-0.06, x=0),
    margin=dict(t=110, b=60),
)
render_plot(fig, width=840, height=760)

Las comunidades salen como zonas reconocibles del grafo, que es lo esperable
porque se calcularon sobre él. La pregunta interesante es otra: **¿ese
agrupamiento coincide con el de K-means, que nunca vio el grafo?** El mapa de
calor lo mide — si cada fila se concentrara en una sola columna, los dos métodos
estarían viendo la misma estructura.

In [27]:
agree = pd.crosstab(df['comunidad'], df['cluster_kmeans'])
fig = go.Figure(go.Heatmap(
    z=agree.values,
    x=[f'Cluster {c}' for c in agree.columns],
    y=[f'Comunidad {i}' for i in agree.index],
    colorscale=[[0, INK['surface']], [1, SEQUENTIAL_BLUE[6]]],
    xgap=2, ygap=2,
    colorbar=dict(title=dict(text='equipos', side='right'), thickness=14, outlinewidth=0),
    hovertemplate='%{y} ∩ %{x}<br>%{z} equipos<extra></extra>',
))
fig.update_layout(
    title=dict(text='¿Coinciden las comunidades con los clusters?',
               subtitle=dict(text='Si cada fila se concentra en una sola columna, los dos métodos '
                                   'están viendo la misma estructura')),
    xaxis=dict(side='top', title=None, showgrid=False),
    yaxis=dict(autorange='reversed', title=None, showgrid=False),
    margin=dict(l=140, t=130),
)
render_plot(fig, width=720, height=460)

## 8. Tabla comparativa final

Comparar los tres métodos tiene una trampa: **las etiquetas no son comparables
entre sí**. El "cluster 0" de K-means no tiene por qué ser el "cluster 0" de
GMM ni la "comunidad 0" de la red, y encima la red eligió su propio número de
grupos. Cualquier verdicto basado en comparar números de etiqueta mide el
etiquetado, no el acuerdo.

La forma correcta de compararlos es preguntar por **compañeros**, que no depende
de cómo se numeró nada:

- **Global**: el *Adjusted Rand Index* entre cada par de particiones — cuánto
  se parecen dos agrupamientos por encima de lo que darían al azar (0 = azar,
  1 = idénticos).
- **Por equipo**: de los compañeros de cluster que le da K-means, **qué fracción
  le confirman también GMM y la red**. Un equipo "coincide" si al menos la mitad
  de su cluster sobrevive a las tres miradas.

La columna **Cluster (GMM)** muestra la etiqueta ya alineada con K-means (solo
para que la tabla se pueda leer de un vistazo); el veredicto no la usa.

In [28]:
from itertools import permutations
from sklearn.metrics import adjusted_rand_score

particiones = {
    'K-means': df['cluster_kmeans'].values,
    'GMM': df['cluster_gmm'].values,
    'Comunidades (red)': df['comunidad'].values,
}
nombres = list(particiones)
ari = pd.DataFrame(
    [[round(adjusted_rand_score(particiones[a], particiones[b]), 2) for b in nombres]
     for a in nombres], index=nombres, columns=nombres)
print('Adjusted Rand Index entre particiones (0 = azar, 1 = idénticas):')
print(ari.to_string())

Adjusted Rand Index entre particiones (0 = azar, 1 = idénticas):
                   K-means   GMM  Comunidades (red)
K-means               1.00  0.69               0.29
GMM                   0.69  1.00               0.32
Comunidades (red)     0.29  0.32               1.00


In [29]:
# alineación SOLO para mostrar: con k=6 son 720 permutaciones, fuerza bruta
# alcanza y se lee mejor que el algoritmo húngaro
best_perm = max(permutations(range(K)),
                key=lambda p: (df['cluster_gmm'].map(dict(enumerate(p))) == df['cluster_kmeans']).sum())
cluster_gmm_alineado = df['cluster_gmm'].map(dict(enumerate(best_perm)))


def companeros(labels):
    """Para cada equipo, el conjunto de índices que comparten su grupo."""
    labels = np.asarray(labels)
    return [set(np.flatnonzero(labels == labels[i])) - {i} for i in range(len(labels))]


comp_km = companeros(df['cluster_kmeans'])
comp_gmm = companeros(df['cluster_gmm'])
comp_red = companeros(df['comunidad'])

nucleo = np.array([len(a & b & c) for a, b, c in zip(comp_km, comp_gmm, comp_red)])
confirmado = np.array([n / max(len(a), 1) for n, a in zip(nucleo, comp_km)])
coinciden = confirmado >= 0.50

print(f'{coinciden.sum()} de {len(df)} equipos ({coinciden.mean():.0%}) conservan al menos la '
      'mitad de su cluster en los tres métodos')
print(f'Núcleo confirmado por equipo: mediana {np.median(nucleo):.0f} compañeros '
      f'(de 0 a {nucleo.max()})')

34 de 96 equipos (35%) conservan al menos la mitad de su cluster en los tres métodos
Núcleo confirmado por equipo: mediana 6 compañeros (de 0 a 11)


In [30]:
comparison = pd.DataFrame({
    'Equipo': df['Squad'],
    'Liga': df['liga'].astype(str),
    'Cluster (k-means)': df['cluster_kmeans'],
    'Cluster (GMM)': cluster_gmm_alineado,
    'Prob. GMM': df['gmm_proba_max'].round(2),
    'Comunidad (red)': df['comunidad'],
    'Núcleo confirmado': nucleo,
    '% del cluster confirmado': (confirmado * 100).round(0).astype(int),
    '¿Coinciden los 3?': np.where(coinciden, 'Sí', 'No'),
}).sort_values(['Liga', 'Equipo']).reset_index(drop=True)

comparison

,Equipo,Liga,Cluster (k-means),Cluster (GMM),Prob. GMM,Comunidad (red),Núcleo confirmado,% del cluster confirmado,¿Coinciden los 3?
0,Augsburg,Bundesliga,2,2,0.93,2,2,8,No
1,Bayern Munich,Bundesliga,4,4,1.00,1,0,0,No
2,Dortmund,Bundesliga,0,0,0.95,1,8,57,Sí
3,Frankfurt,Bundesliga,1,1,0.88,5,1,6,No
4,Freiburg,Bundesliga,2,2,0.82,5,4,15,No
...,...,...,...,...,...,...,...,...,...
91,Pisa,Serie A,3,3,0.98,4,2,17,No
92,Roma,Serie A,0,5,0.75,1,2,14,No
93,Sassuolo,Serie A,5,5,0.94,5,1,6,No
94,Torino,Serie A,2,2,0.95,0,8,31,No


In [31]:
resumen = (pd.DataFrame({'cluster': df['cluster_kmeans'], 'coinciden': coinciden,
                          'nucleo': nucleo, 'confirmado': confirmado * 100})
             .groupby('cluster')
             .agg(Equipos=('coinciden', 'size'), Coinciden=('coinciden', 'sum'),
                  **{'Núcleo medio': ('nucleo', 'mean'), '% medio confirmado': ('confirmado', 'mean')})
             .round(1))
resumen.index = [f'Cluster {c}' for c in resumen.index]
resumen

,Equipos,Coinciden,Núcleo medio,% medio confirmado
Cluster 0,15,9,5.2,37.1
Cluster 1,18,12,7.7,45.1
Cluster 2,27,0,7.0,26.8
Cluster 3,13,8,4.9,41.0
Cluster 4,6,5,3.3,66.7
Cluster 5,17,0,2.4,14.7


### Lectura rápida

- **El acuerdo global es parcial, y hay que decirlo.** K-means y GMM se parecen
  bastante (ARI 0.69) porque comparten supuestos: ambos buscan grupos compactos
  alrededor de un centro. La red baja a 0.29 contra K-means y 0.32 contra GMM.
  No es un fallo — es la señal de que, al no asumir forma esférica ni número de
  grupos, encuentra vecindarios que los otros dos parten al medio. Si los tres
  coincidieran del todo, el tercer método no estaría aportando nada.
- **Los arquetipos más robustos son el 1 y el 4.** En el cluster 1 ("ofensivos
  de presión") coinciden 12 de 18 equipos, y Lens, Lille, Marsella, Mónaco,
  Rennes y Hoffenheim conservan 11 de sus 17 compañeros en los tres métodos — el
  núcleo más grande del estudio. El cluster 4 ("aplanadoras") es el de mayor
  porcentaje confirmado (67%): pese a ser solo 6 equipos, 5 de ellos sobreviven
  a las tres miradas. La élite europea es un grupo real, no un artefacto.
- **Ojo con el sesgo de tamaño del criterio**: exigir que sobreviva la mitad del
  cluster castiga a los grandes. El cluster 2 tiene 27 equipos y ninguno llega al
  50%, pero su núcleo medio (7.0 compañeros confirmados) es de los más altos —
  mayor que el del cluster 3 (4.9), que sí llega al 50% porque solo tiene 13
  miembros. La columna **Núcleo confirmado** es la comparación justa entre
  clusters; el "Sí/No" conviene leerlo dentro de cada cluster, no entre clusters
  de distinto tamaño.
- **El arquetipo más frágil es el 5** ("repliegue con portero salvador"): núcleo
  medio de 2.4 compañeros y 15% confirmado, el más bajo de los seis. Es el grupo
  definido por lo que *no* hace (poco tiro, poca recuperación), y esa clase de
  identidad negativa es la que peor resiste cambiar de método.
- Ser híbrido en GMM y ser inestable no es lo mismo: **Angers** tiene solo 53% en
  su componente más probable y aun así conserva 9 compañeros en los tres
  métodos. Los candidatos a híbrido genuino son los que suman las dos cosas —
  probabilidad baja **y** núcleo chico: Toulouse, Elche, Mallorca, Real Betis y
  Newcastle (núcleo de 0 o 1).
- **Y el recordatorio de la sección 5.1**: todo esto describe un espacio donde el
  eje dominante es el nivel. Que tres métodos coincidan en agrupar a Lens con
  Lille no prueba que jueguen igual — prueba que están igual de bien. La sección
  9 rehace el ejercicio sin esa contaminación.

## 9. Quitando el nivel: ¿queda estilo?

La sección 5.1 mostró dos problemas concretos, así que acá se corrigen los dos y
se vuelve a correr:

**1. Features que miden eficacia, no forma de jugar.** Salen `Goles/90`,
`Asistencias/90`, `Goles por tiro`, `Goles por tiro a puerta`, `Precisión de
tiro`, `Goles recibidos/90` y `Atajadas (%)`: todas responden *qué tan bien le
sale*, no *qué intenta*. Sale también `Rojas/90`, que como se vio en la sección
1 es casi solo ruido. Quedan 10 features de pura conducta: cuánta pelota tiene,
cuánto tira, por dónde ataca, cuánto interrumpe y cuánto le llegan.

**2. `whiten=True` en el PCA.** Así los componentes entran a k-means con la misma
varianza y ninguno manda por escala. Es la corrección de la geometría inclinada.

Ojo con la expectativa: el `Tiros/90` que se queda **también** correlaciona con
nivel (los buenos tiran más). No se trata de eliminar el nivel —es imposible con
estos datos y sería deshonesto fingirlo— sino de que deje de ser *el* eje.

In [32]:
FEATURES_ESTILO = {
    'ov_Poss': 'Posesión (%)',
    'sh_Standard_Sh/90': 'Tiros/90',
    'p90_Crs': 'Centros/90',
    'p90_Off': 'Fuera de juego/90',
    'p90_Fls': 'Faltas cometidas/90',
    'p90_Fld': 'Faltas recibidas/90',
    'p90_CrdY': 'Amarillas/90',
    'p90_Int': 'Intercepciones/90',
    'p90_TklW': 'Tackles ganados/90',
    'p90_SoTA': 'Tiros a puerta recibidos/90',
}
estilo_cols = list(FEATURES_ESTILO)

X_estilo = pd.DataFrame(StandardScaler().fit_transform(df[estilo_cols]),
                         columns=estilo_cols, index=df['Squad'])

pca_e_full = PCA(random_state=RNG).fit(X_estilo)
n_estilo = int(np.searchsorted(np.cumsum(pca_e_full.explained_variance_ratio_), 0.80) + 1)

# whiten=True: cada componente sale con varianza 1, así ninguno domina la
# distancia por tener más varianza que los demás
scores_estilo = PCA(n_components=n_estilo, whiten=True, random_state=RNG).fit_transform(X_estilo)

df['cluster_estilo'] = KMeans(n_clusters=K, random_state=RNG, n_init=10).fit_predict(scores_estilo)

print(f'{len(estilo_cols)} features de conducta -> {n_estilo} componentes '
      f'({np.cumsum(pca_e_full.explained_variance_ratio_)[n_estilo-1]:.0%} de la varianza), blanqueados')
print(f'PC1 ahora explica {pca_e_full.explained_variance_ratio_[0]:.1%} '
      f'(antes {explained[0]:.1%}) — ya no hay un eje que aplaste al resto')

10 features de conducta -> 5 componentes (80% de la varianza), blanqueados
PC1 ahora explica 31.1% (antes 38.2%) — ya no hay un eje que aplaste al resto


In [33]:
pred_e = df.groupby('cluster_estilo', observed=True)['pt_Team Success_PPM'].transform('mean')
eta2_estilo = 1 - ((ppm - pred_e) ** 2).sum() / ((ppm - ppm.mean()) ** 2).sum()

print('¿Cuánto siguen siendo "la tabla de posiciones"?')
print(f'  Sección 5 (19 features, sin blanquear): {eta2_ppm:.0%} de la variación en puntos')
print(f'  Sección 9 (10 features, blanqueado)   : {eta2_estilo:.0%}')
print(f'\ncorr(PC1 de estilo, puntos por partido) = '
      f'{np.corrcoef(scores_estilo[:, 0], ppm)[0, 1]:+.2f}  (antes +0.93)')
print(f'\nSilhouette: {silhouette_score(scores_estilo, df["cluster_estilo"]):.3f} '
      f'(antes {silhouette_score(scores, df["cluster_kmeans"]):.3f}) — sigue siendo un continuo, '
      'como mostró el test de nulo')

¿Cuánto siguen siendo "la tabla de posiciones"?
  Sección 5 (19 features, sin blanquear): 70% de la variación en puntos
  Sección 9 (10 features, blanqueado)   : 50%

corr(PC1 de estilo, puntos por partido) = +0.81  (antes +0.93)

Silhouette: 0.174 (antes 0.161) — sigue siendo un continuo, como mostró el test de nulo


In [34]:
profile_e = X_estilo.groupby(df['cluster_estilo'].values).mean()
profile_e.index = [f'Estilo {c}' for c in profile_e.index]

fig = go.Figure(go.Heatmap(
    z=profile_e.T.values, x=profile_e.index,
    y=[FEATURES_ESTILO[f] for f in estilo_cols],
    colorscale=DIVERGING_SCALE, zmid=0, xgap=2, ygap=2,
    colorbar=dict(title=dict(text='z-score', side='right'), thickness=14, outlinewidth=0),
    hovertemplate='%{x}<br>%{y}<br>%{z:+.2f} desviaciones vs. el promedio<extra></extra>',
))
fig.update_layout(
    title=dict(text='Los grupos de estilo, ya sin el eje de nivel',
               subtitle=dict(text='Mismas 6 divisiones, pero definidas por conducta: '
                                   'dónde ataca, cuánto interrumpe, cuánto le llegan')),
    xaxis=dict(side='top', title=None, showgrid=False),
    yaxis=dict(autorange='reversed', title=None, showgrid=False),
    margin=dict(l=200, t=155),
)
render_plot(fig, width=760, height=520)

In [35]:
for c in sorted(df['cluster_estilo'].unique()):
    sub = df[df['cluster_estilo'] == c]
    print(f'--- Estilo {c} ({len(sub)} equipos · {sub["pt_Team Success_PPM"].mean():.2f} ppm medio) ---')
    print(', '.join(sub['Squad']))
    print()

--- Estilo 0 (15 equipos · 1.27 ppm medio) ---
Augsburg, Mainz 05, Le Havre, Monaco, Paris FC, Rennes, Alavés, Espanyol, Getafe, Oviedo, Rayo Vallecano, Real Sociedad, Sevilla, Bournemouth, Chelsea

--- Estilo 1 (16 equipos · 1.62 ppm medio) ---
Dortmund, Hoffenheim, Atalanta, Inter, Juventus, Lens, Lille, Lyon, Marseille, Athletic Club, Atlético Madrid, Osasuna, Brighton, Newcastle, Nottingham, Tottenham

--- Estilo 2 (8 equipos · 2.21 ppm medio) ---
Bayern Munich, RB Leipzig, PSG, Barcelona, Real Madrid, Arsenal, Manchester City, Manchester Utd

--- Estilo 3 (18 equipos · 1.40 ppm medio) ---
Freiburg, Hamburger SV, Stuttgart, Bologna, Cremonese, Fiorentina, Milan, Napoli, Parma, Sassuolo, Lorient, Strasbourg, Celta Vigo, Girona, Mallorca, Valencia, Aston Villa, Liverpool

--- Estilo 4 (24 equipos · 1.03 ppm medio) ---
Frankfurt, Gladbach, Heidenheim, Köln, Leverkusen, St Pauli, Union Berlin, Werder Bremen, Wolfsburg, Pisa, Torino, Angers, Metz, Nice, Levante, Brentford, Burnley, Crys

### Los seis estilos, ahora sí

| Grupo | Nombre | Qué lo define | Ejemplos |
|---|---|---|---|
| **2** (8) | **Dominio central** | Posesión y tiros por las nubes (+1.7, +2.0) pero **centros por debajo del promedio** (−0.6) y las menos faltas y amarillas de todos: atacan por dentro y casi no tienen que interrumpir. | Bayern, PSG, Madrid, City, Arsenal |
| **1** (16) | **Ataque por banda** | **Los más centros del estudio** (+1.3) con buena posesión y volumen de tiro, y pocos tiros a puerta en contra. Control, pero llegando por fuera. | Inter, Juventus, Lens, Lille, Atlético, Newcastle |
| **0** (15) | **Vertical y áspero** | **Los más fuera de juego (+1.3) y las más amarillas (+1.1)**, con faltas y duelos altos. Ataque de ruptura a la espalda, defensa a base de interrumpir. | Getafe, Rayo, Sevilla, Chelsea, Mónaco |
| **5** (15) | **Trabado, sin bandas** | Faltas altas en los dos sentidos (+0.8 cometidas, +0.8 recibidas) y **los menos centros (−0.9) y menos fuera de juego (−0.8)**: partido cortado, poco juego directo y poco por fuera. | Roma, Lazio, Genoa, Verona, Betis, Villarreal |
| **3** (18) | **Sin duelos** | **Los menos tackles (−1.2) e intercepciones (−0.9)** del estudio, y reciben muchas faltas. Ni presionan ni cortan: dejan jugar y esperan. | Napoli, Milan, Liverpool, Fiorentina, Freiburg |
| **4** (24) | **Replegados y expuestos** | **Los más tiros a puerta en contra (+0.9)** con la menor posesión y volumen de tiro, y sin compensarlo con faltas. El bloque que aguanta. | Everton, Burnley, Union Berlin, Torino, Leeds |

Ahora los nombres describen **conducta**, no jerarquía. Y se nota en el detalle
que antes era imposible: el grupo *Sin duelos* junta a Liverpool, Napoli y Milan
con Freiburg y Cremonese — equipos de niveles muy distintos que comparten una
forma de no defender. En la sección 5 eso no podía pasar, porque el nivel los
mandaba a clusters distintos antes de mirarles el estilo.

In [36]:
cruce = pd.crosstab(df['cluster_kmeans'], df['cluster_estilo'])
cruce.index = [f'Cluster {c} (sec. 5)' for c in cruce.index]
cruce.columns = [f'Estilo {c}' for c in cruce.columns]
print('¿Se parecen las dos particiones?')
print(cruce.to_string())
print(f'\nARI entre la partición de la sección 5 y la de estilo: '
      f'{adjusted_rand_score(df["cluster_kmeans"], df["cluster_estilo"]):.2f}')
print('Bajo = quitar el nivel reorganizó de verdad a los equipos, no fue un retoque.')

¿Se parecen las dos particiones?
                    Estilo 0  Estilo 1  Estilo 2  Estilo 3  Estilo 4  Estilo 5
Cluster 0 (sec. 5)         0         6         2         4         1         2
Cluster 1 (sec. 5)         5         7         1         1         1         3
Cluster 2 (sec. 5)         3         1         0         3        18         2
Cluster 3 (sec. 5)         7         1         0         0         2         3
Cluster 4 (sec. 5)         0         0         5         1         0         0
Cluster 5 (sec. 5)         0         1         0         9         2         5

ARI entre la partición de la sección 5 y la de estilo: 0.23
Bajo = quitar el nivel reorganizó de verdad a los equipos, no fue un retoque.


## Conclusión

Tres cosas que se lleva este notebook, en orden de importancia:

1. **Con datos públicos de fbref, el eje dominante del fútbol europeo es el
   nivel, no el estilo.** No hizo falta creerlo: el primer componente correlaciona
   `+0.93` con unos puntos por partido que nunca entraron al modelo, y los
   clusters explicaban el 70% de su variación. Coincide con la intuición de
   cualquiera que mire estas ligas —los buenos son muy buenos y eso tiñe todas
   las métricas— pero acá está medido.
2. **No hay arquetipos naturales, hay un continuo.** El test de nulo mostró que
   el silhouette real es indistinguible del de una nube sin grupos. Los seis
   grupos son una segmentación útil, no tribus con fronteras.
3. **Quitando eficacia y blanqueando el PCA, sí aparece estilo** — el peso del
   nivel baja del 70% al 50% y emergen ejes reconocibles: atacar por dentro o por
   banda, buscar la espalda o cortar con falta, presionar o dejar jugar.

**Qué haría falta para ir más lejos**: métricas de acción y ubicación (xG, pases
progresivos, altura de recuperación, PPDA) que estos CSV gratuitos no traen. Con
ellas el estilo dejaría de tener que inferirse desde faltas y centros. Y para
afirmar que un arquetipo es estable, remuestreo tipo bootstrap: ver qué grupos se
reproducen y cuáles se deshacen al cambiar los datos de entrada.

In [37]:
export = pd.DataFrame({
    'Squad': df['Squad'],
    'liga': df['liga'].astype(str),
    'ppm': df['pt_Team Success_PPM'],
    'cluster_kmeans': df['cluster_kmeans'],
    'cluster_gmm': df['cluster_gmm'],
    'cluster_gmm_alineado': cluster_gmm_alineado,
    'gmm_proba_max': df['gmm_proba_max'].round(3),
    'gmm_hibrido': df['gmm_hibrido'],
    'comunidad': df['comunidad'],
    'nucleo_confirmado': nucleo,
    'pct_cluster_confirmado': (confirmado * 100).round(0).astype(int),
    'coinciden_3': coinciden,
    'cluster_estilo': df['cluster_estilo'],
})
export.to_csv('../data/processed/style_clusters.csv', index=False, encoding='utf-8-sig')
print(f'Guardado en data/processed/style_clusters.csv ({len(export)} equipos, {export.shape[1]} columnas)')
print('`cluster_kmeans` = partición contaminada por nivel (secciones 5-8)')
print('`cluster_estilo` = partición de conducta, la buena para hablar de estilo (sección 9)')

Guardado en data/style_clusters.csv (96 equipos, 13 columnas)
`cluster_kmeans` = partición contaminada por nivel (secciones 5-8)
`cluster_estilo` = partición de conducta, la buena para hablar de estilo (sección 9)
